# 00_setup

### Obiettivo
Notebook per la configurazione iniziale del progetto, il caricamento dei dataset e la verifica della struttura dei file.

### Dataset utilizzato
MovieLens-25M

### Struttura del progetto
- data/raw/: dati originali
- data/processed/: dati elaborati
- notebooks/: notebook di analisi
- src/: script Python
- outputs/: grafici e risultati
- report/: relazione finale

### Test ambiente:

In [1]:
import sys
print(sys.executable)

/Users/sofiadimaggio/opt/anaconda3/envs/movielens/bin/python


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("Ambiente configurato correttamente")

Ambiente configurato correttamente


### Controllo cartella progetto

In [2]:
# Individua la cartella principale del progetto
cwd = Path.cwd()

# Se il notebook viene eseguito dalla cartella notebooks, risale alla cartella principale
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd

print("Cartella di lavoro:", cwd)
print("Cartella progetto:", project_root)

Cartella di lavoro: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/notebooks
Cartella progetto: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens


### Definizione percorsi

In [3]:
data_raw = project_root / "data" / "raw"
data_processed = project_root / "data" / "processed"
outputs = project_root / "outputs"

print("Cartella dati originali:", data_raw)
print("Cartella dati elaborati:", data_processed)
print("Cartella output:", outputs)

Cartella dati originali: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/data/raw
Cartella dati elaborati: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/data/processed
Cartella output: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/outputs


### Controllo presenza dataset

In [4]:
movielens_path = data_raw / "ml-25m"

if movielens_path.exists():
    print("Dataset MovieLens trovato.")
    print("File presenti:")
    for file in movielens_path.iterdir():
        print("-", file.name)
else:
    print("Dataset MovieLens non ancora presente.")
    print("Scaricare MovieLens-25M e inserirlo in:", movielens_path)


Dataset MovieLens trovato.
File presenti:
- links.csv
- tags.csv
- genome-tags.csv
- ratings.csv
- README.txt
- genome-scores.csv
- movies.csv


### Controllo file obbligatori

In [5]:
required_files = ["ratings.csv", "movies.csv", "links.csv"]

missing_files = []

for file in required_files:
    if not (movielens_path / file).exists():
        missing_files.append(file)

if missing_files:
    print("File mancanti:")
    for file in missing_files:
        print("-", file)
else:
    print("Tutti i file principali sono presenti.")

Tutti i file principali sono presenti.


### Controllo caricamento database

In [6]:
ratings = pd.read_csv(movielens_path / "ratings.csv")
movies = pd.read_csv(movielens_path / "movies.csv")
links = pd.read_csv(movielens_path / "links.csv")

print("ratings:", ratings.shape)
print("movies:", movies.shape)
print("links:", links.shape)

display(ratings.head())
display(movies.head())
display(links.head())

ratings: (25000095, 4)
movies: (62423, 3)
links: (62423, 3)


,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


### Controllo colonne

In [7]:
print("Ratings columns:", ratings.columns.tolist())
print("Movies columns:", movies.columns.tolist())
print("Links columns:", links.columns.tolist())

Ratings columns: ['userId', 'movieId', 'rating', 'timestamp']
Movies columns: ['movieId', 'title', 'genres']
Links columns: ['movieId', 'imdbId', 'tmdbId']


### Dimensione ratings.csv

In [8]:
import os

ratings_size = os.path.getsize(movielens_path / "ratings.csv") / (1024**2)

print(f"Dimensione ratings.csv: {ratings_size:.2f} MB")

Dimensione ratings.csv: 646.84 MB


### IMDb mapping

In [9]:
links["imdbId_str"] = links["imdbId"].astype(int).apply(lambda x: f"tt{x:07d}")

display(links[["movieId", "imdbId", "imdbId_str"]].head())

,movieId,imdbId,imdbId_str
0,1,114709,tt0114709
1,2,113497,tt0113497
2,3,113228,tt0113228
3,4,114885,tt0114885
4,5,113041,tt0113041


### Salvataggio parquet

In [10]:
imdb_mapping = links[["movieId", "imdbId_str"]].copy()

output_path = data_processed / "imdb_mapping.parquet"
imdb_mapping.to_parquet(output_path, index=False)

print("Mapping IMDb salvato in:", output_path)

Mapping IMDb salvato in: /Users/sofiadimaggio/Desktop/uni/3 anno/Tirocinio_MovieLens/data/processed/imdb_mapping.parquet
